In [1]:
import torch
import clip
from PIL import Image
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_image


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)

# Text prompts for CLIP
text_prompts = ["a photo of a leaf", "a photo of not a leaf"]
text_tokens = clip.tokenize(text_prompts).to(device)

In [ ]:
disease_model = load_model('plants_disease_model_best.h5')

class_labels = {
    0: "Pepper_healthy",
    1: "Pepper_leaf_blight",
    2: "Pepper_yellow_mottle_virus",
    3: "Rice BrownSpot",
    4: "Rice Healthy",
    5: "Rice Hispa",
    6: "Rice LeafBlast",
    7: "Sugarcane healthy",
    8: "Sugarcane mosaic",
    9: "Sugarcane red rot",
    10: "Sugarcane rust",
    11: "Sugarcane yellow leaf",
    12: "Tomato Bacterial Spot",
    13: "Tomato Early Blight",
    14: "Tomato Healthy",
    15: "Tomato Late Blight",
    16: "Tomato Septoria Leaf Spot",
    17: "Tomato Yellow Leaf Curl Virus",
    18: "Tomato leaf mold",
    19: "Tomato mosaic virus",
    20: "Tomato spider mites two-spotted spider mite",
    21: "Tomato target spot"
}

In [4]:
def is_leaf_image(img_path, threshold=0.5):
    img = preprocess(Image.open(img_path)).unsqueeze(0).to(device)

    with torch.no_grad():
        logits_per_image, _ = clip_model(img, text_tokens)
        probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]

    print(f"Leaf Probability: {probs[0]:.4f}, Non-Leaf Probability: {probs[1]:.4f}")
    return probs[0] > threshold  # True if leaf probability > threshold

In [5]:
def predict_disease(img_path):
    img = keras_image.load_img(img_path, target_size=(150, 150))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = disease_model.predict(img_array)
    predicted_class_idx = np.argmax(prediction, axis=1)[0]
    predicted_class = class_labels.get(predicted_class_idx, "Unknown Disease")

    print(f"Predicted Disease: {predicted_class}")



In [ ]:
image_path = r"training_history.png"

if is_leaf_image(image_path):
    print("This is a leaf image. Proceeding to disease detection...")
    predict_disease(image_path)
else:
    print("This is not a leaf image. Skipping disease detection.")


Leaf Probability: 0.6278, Non-Leaf Probability: 0.3722
This is a leaf image. Proceeding to disease detection...
1/1 [==============================] - 0s 25ms/step
Predicted Disease: Rice BrownSpot
